# The Architecture of Data Integrity — Project 1: Data Cleaning & Preparation

**Intern Project:** DecodeLabs Internship  
**Goal:** Clean a raw e-commerce orders dataset by handling missing values, removing duplicates, and correcting data formats — with every change logged and verified.

**Verification Gate (must pass):**
- ✅ 0% duplicate unique identifiers (OrderID)
- ✅ 0% incorrectly formatted dates

This notebook follows the three-phase approach from the training material:
1. **Strategic Imputation** — handle missing values without deleting rows
2. **The Integrity Audit** — eliminate duplicate records
3. **Speak One Language** — standardize dates, numbers, and text formatting

## 0. Setup & Load Raw Data

In [1]:
import numpy as np
import pandas as pd

# The raw file is tab-separated, not comma-separated - confirmed during initial inspection
dataset = pd.read_csv("dataset.csv", sep="\t")

In [2]:
# Inspect structure: row/column counts, dtypes, non-null counts
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   str    
 1   Date             1200 non-null   str    
 2   CustomerID       1200 non-null   str    
 3   Product          1200 non-null   str    
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   str    
 7   PaymentMethod    1200 non-null   str    
 8   OrderStatus      1200 non-null   str    
 9   TrackingNumber   1200 non-null   str    
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       891 non-null    str    
 12  ReferralSource   1200 non-null   str    
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), str(10)
memory usage: 131.4 KB


In [3]:
dataset.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='str')

## 1. Identify Missing Values

Before changing anything, we first quantify the problem across every column.

In [4]:
missing_before = dataset.isnull().sum()
print("Missing values per column (before cleaning):")
print(missing_before[missing_before > 0])

Missing values per column (before cleaning):
CouponCode    309
dtype: int64


**Finding:** `CouponCode` has 309 missing values (25.75% of rows). Every other column is fully populated.

**Decision:** A missing `CouponCode` almost certainly means *no coupon was applied* to that order, not a data entry error. Following the training material's guidance to avoid listwise deletion (which reduces statistical power), we impute this with a meaningful category — `"No Coupon"` — rather than deleting rows or using a vague placeholder like the string `"Nan"`.

## 2. Phase 1 — Strategic Imputation: Fix Missing Values

In [5]:
# Fill missing CouponCode with a meaningful category instead of deleting rows
# or using a vague "Nan" string that could be mistaken for real missing-data markers
dataset['CouponCode'] = dataset['CouponCode'].fillna('No Coupon')

print("Missing values remaining:", dataset.isnull().sum().sum())

Missing values remaining: 0


## 3. Phase 2 — The Integrity Audit: Check & Remove Duplicates

Every `OrderID` should represent exactly one unique transaction. We check for duplicates on:
- **Full rows** (every column identical)
- **OrderID specifically** (the unique identifier — equivalent to `GROUP BY OrderID HAVING COUNT(*) > 1`)

In [6]:
rows_before = len(dataset)

full_duplicates = dataset.duplicated().sum()
orderid_duplicates = dataset['OrderID'].duplicated().sum()

print("Full duplicate rows found:", full_duplicates)
print("Duplicate OrderIDs found:", orderid_duplicates)

Full duplicate rows found: 0
Duplicate OrderIDs found: 0


In [7]:
# Remove any duplicate OrderIDs, keeping the first occurrence
# (No duplicates were found in this dataset, but this step is kept to make the
# pipeline robust and reproducible against any future/updated version of the data)
dataset = dataset.drop_duplicates(subset='OrderID', keep='first')

rows_after_dedup = len(dataset)
print("Rows removed as duplicates:", rows_before - rows_after_dedup)

Rows removed as duplicates: 0


## 4. Phase 3 — Speak One Language: Standardize Formats

### 4a. Dates → ISO 8601 (`YYYY-MM-DD`)

In [8]:
# errors='coerce' turns any unparseable date into NaT instead of crashing,
# so we can explicitly detect and report bad dates rather than missing them silently
dataset['Date'] = pd.to_datetime(dataset['Date'], errors='coerce').dt.strftime('%Y-%m-%d')

invalid_dates = dataset['Date'].isnull().sum()
print("Unparseable/invalid dates after standardization:", invalid_dates)

Unparseable/invalid dates after standardization: 0


### 4b. Text → Trimmed Whitespace + Consistent Case

Applied to display-text columns (categories, addresses). `CouponCode` is a **code**, not 
display text, so it's only trimmed - not title-cased - to preserve its original format 
(e.g. `SAVE10` stays `SAVE10`, not `Save10`).

In [9]:
display_text_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'ShippingAddress']

for col in display_text_cols:
    dataset[col] = dataset[col].str.strip().str.title()

# CouponCode: trim only, preserve original casing since it's a code, not display text
dataset['CouponCode'] = dataset['CouponCode'].str.strip()

print("Text standardization applied to:", display_text_cols + ['CouponCode (trim only)'])

Text standardization applied to: ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'ShippingAddress', 'CouponCode (trim only)']


### 4c. Numbers → Consistent Precision (2 decimal places)

In [10]:
dataset['UnitPrice'] = dataset['UnitPrice'].round(2)
dataset['TotalPrice'] = dataset['TotalPrice'].round(2)

print("Numeric columns rounded to 2 decimal places: UnitPrice, TotalPrice")

Numeric columns rounded to 2 decimal places: UnitPrice, TotalPrice


## 5. Verification Gate

Before this dataset is considered production-ready, we must prove:
- **0% duplicate unique identifiers**
- **0% incorrectly formatted dates**

In [11]:
print("=== VERIFICATION GATE ===")
print("Total missing values:", dataset.isnull().sum().sum())
print("Duplicate OrderIDs:", dataset['OrderID'].duplicated().sum(),
      f"({dataset['OrderID'].duplicated().sum() / len(dataset) * 100:.2f}%)")
print("Invalid/unparseable dates:", dataset['Date'].isnull().sum(),
      f"({dataset['Date'].isnull().sum() / len(dataset) * 100:.2f}%)")
print("Final row count:", len(dataset))
print()
print("Sample of cleaned data:")
dataset.head(5)

=== VERIFICATION GATE ===
Total missing values: 0
Duplicate OrderIDs: 0 (0.00%)
Invalid/unparseable dates: 0 (0.00%)
Final row count: 1200

Sample of cleaned data:


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 6. Save Cleaned Dataset

In [12]:
dataset.to_csv('cleaned_dataset.csv', index=False)
print("Saved cleaned_dataset.csv - shape:", dataset.shape)

Saved cleaned_dataset.csv - shape: (1200, 14)
